# Research Question 7 – Final Model Benchmark  
_Auto‑generated notebook – 2025-06-08 09:48 UTC_  

This notebook assembles the **four best-performing CNN variants** discovered in previous experiments and compares their performance on:

* **Nickparvar** – in‑domain validation set  
* **BraTS 2023** – external-domain test set  

The models are:

| Abbrev. | Variant | Purpose |
|---------|---------|---------|
| **RN18‑TL** | ResNet‑18 fine‑tuned from ImageNet | Strong transfer‑learning baseline |
| **SCNN‑4**  | 4‑conv SimpleCNN | Lightweight custom baseline |
| **DH‑CNN**  | Double‑Head CNN (binary + three‑class heads) | Detects tumour presence **and** subtype |
| **LpCNN**   | CNN with learnable $L_p$ masks | Biologically inspired receptive fields |

> **Preprocessing:** 224 × 224 px resize · global $z$‑score · CLAHE  
> **Augmentation (training only):** Horizontal flip $p=0.5$  
> **Class imbalance:** oversample + weighted CE  
> **Optimiser:** Adam ($\eta = 1\times10^-3$)  

Set the variable `EPOCHS` in the **Hyper‑parameters** section to control training length.


## 0 · Dependencies & Environment

Install project requirements (if not already fulfilled) and import core libraries.


In [5]:
import os, sys, random, time, itertools, copy, math, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score

# project modules -----------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent    # adjust if notebook placed elsewhere
SRC_DIR = PROJECT_ROOT / 'src'
if SRC_DIR.as_posix() not in sys.path:
    sys.path.append(SRC_DIR.as_posix())

from dataset import BrainTumorDataset
from models import SimpleCNN
from lpconv import LpCNN, init_lp_weights

import torchvision.transforms as T
import torchvision.models as models
import torchvision.transforms.functional as F
import cv2
from PIL import Image

# reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Running on', DEVICE)


ModuleNotFoundError: No module named 'src'

## 1 · Datasets & Transforms

* **Nickparvar** directory structure (as downloaded from Kaggle):
```
data/
└── archive/
    ├── Training/
    │   ├── Glioma/
    │   ├── Meningioma/
    │   ├── NoTumor/
    │   └── Pituitary/
    └── Testing/
        └── ...
```
* **BraTS 2023** – pre‑extracted axial slices placed in  
  `data/brats_slices/` with **`glioma/`** sub‑folder.

Adjust the two path variables below if your folder names differ.


In [ ]:
# ------------------------------------------------------------------
# Configurable paths
# ------------------------------------------------------------------
NICK_ROOT = PROJECT_ROOT / 'data' / 'archive'        # Nickparvar root
BRATS_ROOT = PROJECT_ROOT / 'data' / 'brats_slices'  # BraTS slices root

assert NICK_ROOT.exists(), f'Nickparvar folder not found: {NICK_ROOT}'
assert BRATS_ROOT.exists(), f'BraTS folder not found: {BRATS_ROOT}'

# ------------------------------------------------------------------
# Pre‑processing transform (resize → CLAHE → ToTensor → z‑score)
# ------------------------------------------------------------------
def clahe_pil(img):
    arr = np.array(img, dtype=np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    arr_eq = clahe.apply(arr)
    return Image.fromarray(arr_eq)

IMG_SIZE = 224
def build_preprocess(zscore=True, clahe=True):
    steps = []
    if clahe:
        steps.append(T.Lambda(clahe_pil))
    steps.append(T.Resize((IMG_SIZE, IMG_SIZE)))
    steps.append(T.ToTensor())
    if zscore:
        steps.append(T.Lambda(lambda x: (x - x.mean()) / (x.std() + 1e-7)))
    return T.Compose(steps)

BASE_PREPROC = build_preprocess()

# training = base + flip aug
TRAIN_TRANSFORM = T.Compose([
    T.RandomHorizontalFlip(),
    *BASE_PREPROC.transforms          # unpack inner steps
])

VAL_TRANSFORM = BASE_PREPROC


### 1.1 Data loaders

In [ ]:
BATCH_SIZE = 16

def make_nick_loaders(batch=BATCH_SIZE):
    train_set = BrainTumorDataset(NICK_ROOT, subset='Training',
                                  transform=TRAIN_TRANSFORM)
    val_set   = BrainTumorDataset(NICK_ROOT, subset='Testing',
                                  transform=VAL_TRANSFORM)

    train_ld = DataLoader(train_set, batch_size=batch, shuffle=True,
                          num_workers=4, pin_memory=True)
    val_ld   = DataLoader(val_set,   batch_size=batch, shuffle=False,
                          num_workers=4, pin_memory=True)
    return train_ld, val_ld

def make_brats_loader(batch=BATCH_SIZE):
    brats_set = BrainTumorDataset(BRATS_ROOT, subset='glioma',
                                  transform=VAL_TRANSFORM,
                                  single_class=True)   # custom arg – adjust
    return DataLoader(brats_set, batch_size=batch, shuffle=False,
                      num_workers=4, pin_memory=True)


## 2 · Model Architectures

In [ ]:
# -------------------- Double‑Head CNN ------------------------------
class DoubleHeadCNN(nn.Module):
    """Shared backbone + binary and three‑class heads."""
    def __init__(self, backbone: nn.Module, feat_dim: int):
        super().__init__()
        self.backbone = backbone          # outputs pooled feature vector
        self.head_bin = nn.Linear(feat_dim, 2)
        self.head_tri = nn.Linear(feat_dim, 3)

    def forward(self, x, mode='four'):
        feat = self.backbone(x)
        if mode == 'binary':
            return self.head_bin(feat)
        elif mode == 'three':
            return self.head_tri(feat)
        elif mode == 'four':
            # concatenate logits (no‑tumour vs tumour subtype)
            bin_logits = self.head_bin(feat)
            tri_logits = self.head_tri(feat)
            # soft protocol: argmax over concatenated logits
            tumor_prob = torch.softmax(tri_logits, 1)
            return torch.cat([bin_logits[:, :1],           # no‑tumour logit
                              bin_logits[:, 1:2] * tumor_prob], dim=1)
        else:
            raise ValueError(mode)


## 3 · Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, *, head_mode=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        if head_mode == 'double':
            logits = model(x, mode='four')
        else:
            logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, *, head_mode=None):
    model.eval()
    loss, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if head_mode == 'double':
            logits = model(x, mode='four')
        else:
            logits = model(x)
        loss += criterion(logits, y).item() * x.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        y_true.extend(y.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
    acc = correct / total
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    return loss / total, acc, macro_f1


## 4 · Hyper‑parameters

In [ ]:
EPOCHS = 10          # ← adjust here for quick tests or longer runs
LR      = 1e-3
WEIGHT_DECAY = 1e-4


## 5 · Train All Models

In [ ]:
results = []
train_loader, val_loader = make_nick_loaders()
brats_loader = make_brats_loader()

criterion = nn.CrossEntropyLoss()

# ---------- Model A: SimpleCNN‑4 ---------------------------------
scnn = SimpleCNN(num_classes=4, input_size=IMG_SIZE, dropout=0.0).to(DEVICE)
opt = optim.Adam(scnn.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_acc = 0.0
for ep in range(EPOCHS):
    train_one_epoch(scnn, train_loader, criterion, opt)
    _, acc, f1 = evaluate(scnn, val_loader, criterion)
    best_acc = max(best_acc, acc)
print(f'SCNN‑4  best Nickparvar acc = {best_acc*100:.2f}%')
val_loss, val_acc, val_f1 = evaluate(scnn, brats_loader, criterion)
results.append(dict(model='SCNN‑4', nick_acc=best_acc, brats_acc=val_acc,
                    brats_f1=val_f1))

# ---------- Model B: ResNet‑18 TL --------------------------------
rn18 = models.resnet18(weights='IMAGENET1K_V1')
rn18.fc = nn.Linear(rn18.fc.in_features, 4)
rn18 = rn18.to(DEVICE)

# phase 1 — train fc only
for p in rn18.parameters(): p.requires_grad = False
for p in rn18.fc.parameters(): p.requires_grad = True
opt = optim.Adam(rn18.fc.parameters(), lr=LR)
for ep in range(1):                          # single epoch
    train_one_epoch(rn18, train_loader, criterion, opt)

# phase 2 — fine‑tune whole net
for p in rn18.parameters(): p.requires_grad = True
opt = optim.Adam(rn18.parameters(), lr=LR*0.1, weight_decay=WEIGHT_DECAY)
best_acc = 0.0
for ep in range(EPOCHS-1):
    train_one_epoch(rn18, train_loader, criterion, opt)
    _, acc, f1 = evaluate(rn18, val_loader, criterion)
    best_acc = max(best_acc, acc)
print(f'ResNet‑18 best Nickparvar acc = {best_acc*100:.2f}%')
val_loss, val_acc, val_f1 = evaluate(rn18, brats_loader, criterion)
results.append(dict(model='RN18‑TL', nick_acc=best_acc, brats_acc=val_acc,
                    brats_f1=val_f1))

# ---------- Model C: LpCNN ---------------------------------------
lp_model = LpCNN(num_classes=4).to(DEVICE)
lp_model.apply(init_lp_weights)
opt = optim.Adam(lp_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_acc = 0.0
for ep in range(EPOCHS):
    train_one_epoch(lp_model, train_loader, criterion, opt)
    _, acc, f1 = evaluate(lp_model, val_loader, criterion)
    best_acc = max(best_acc, acc)
print(f'LpCNN    best Nickparvar acc = {best_acc*100:.2f}%')
val_loss, val_acc, val_f1 = evaluate(lp_model, brats_loader, criterion)
results.append(dict(model='LpCNN', nick_acc=best_acc, brats_acc=val_acc,
                    brats_f1=val_f1))

# ---------- Model D: Double‑Head CNN ------------------------------
# shared backbone = SimpleCNN feature extractor (remove final FC layers)
backbone = SimpleCNN(num_classes=4, input_size=IMG_SIZE).features
feat_dim = 16 *  (IMG_SIZE // 8) ** 2     # adjust if you change CNN
backbone = nn.Sequential(backbone, nn.Flatten())
dh_cnn = DoubleHeadCNN(backbone, feat_dim=feat_dim).to(DEVICE)

opt = optim.Adam(dh_cnn.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_acc = 0.0
for ep in range(EPOCHS):
    train_one_epoch(dh_cnn, train_loader, criterion, opt, head_mode='double')
    _, acc, f1 = evaluate(dh_cnn, val_loader, criterion, head_mode='double')
    best_acc = max(best_acc, acc)
print(f'DH‑CNN   best Nickparvar acc = {best_acc*100:.2f}%')
val_loss, val_acc, val_f1 = evaluate(dh_cnn, brats_loader, criterion,
                                     head_mode='double')
results.append(dict(model='DH‑CNN', nick_acc=best_acc, brats_acc=val_acc,
                    brats_f1=val_f1))

results_df = pd.DataFrame(results)
display(results_df)


## 6 · Discussion

* **ResNet‑18** is expected to lead on in‑domain accuracy but incurs the heaviest memory and compute cost.  
* **SCNN‑4** remains a competitive baseline that trains quickly.  
* **LpCNN** may benefit small, irregular tumours thanks to flexible receptive masks, though convergence can be unstable.  
* **DH‑CNN** offers a clinically plausible two‑stage output (tumour presence & subtype), with only a minor performance penalty versus the four‑class head.

External‑domain BraTS results quantify generalisation gaps.  
Consider fine‑tuning on a small fraction of BraTS data or employing self‑supervised pre‑training to narrow the gap.
